In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EmpApp").getOrCreate()

Employee DataSet

In [2]:
data = [
(101,"Alice","HR",50000,"Pune",28,"2021-03-15","Male"),
(102,"Bob","IT",60000,"Mumbai",32,"2020-07-10","Male"),
(103,"Cathy","HR",55000,"Pune",30,"2022-01-05","Female"),
(104,"David","IT",70000,"Bangalore",35,"2019-11-20","Female"),
(105,"Eva","Finance",65000,"Mumbai",29,"2021-06-18","Male"),
(106,"Frank","IT",72000,"Pune",33,"2018-09-12",None),
(107,"Grace","Finance",68000,"Bangalore",31,"2020-12-01","Female"),
(108,"Henry","HR",52000,"Mumbai",27,"2023-04-22","Male")
]
columns = ["emp_id","name","department","salary","city","age","hire_date","gender"]

In [3]:
emp_df = spark.createDataFrame(data, columns)

In [4]:
emp_df.show()

+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  NULL|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
+------+-----+----------+------+---------+---+----------+------+



In [5]:
emp_df.rdd.getNumPartitions()

2

In [6]:
emp_df.printSchema()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- gender: string (nullable = true)



In [7]:
#filtering 1

#emp_filtered = emp_df.filter(emp_df.salary > 50000).select("emp_id","name","salary","department")
#or
emp_filtered = emp_df.where((emp_df.salary > 50000) & (emp_df.gender == 'Female')).select("emp_id","name","salary","department","gender")

In [8]:
emp_filtered.count()

3

In [9]:
emp_filtered.show()

+------+-----+------+----------+------+
|emp_id| name|salary|department|gender|
+------+-----+------+----------+------+
|   103|Cathy| 55000|        HR|Female|
|   104|David| 70000|        IT|Female|
|   107|Grace| 68000|   Finance|Female|
+------+-----+------+----------+------+



In [10]:
#filtering 2
from pyspark.sql.functions import expr,col
#emp_filtered1 = emp_df.select(expr("age > 25 as age"),expr("cast(emp_id as int) as emp_id"),expr("name"),expr("salary"))
#or
emp_filtered1 = emp_df.filter("age > 25")\
                      .select(
                          expr("cast(emp_id as int) as emp_id"),
                          expr("name"),
                          expr("salary"))

In [11]:
emp_filtered1.show()

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|   101|Alice| 50000|
|   102|  Bob| 60000|
|   103|Cathy| 55000|
|   104|David| 70000|
|   105|  Eva| 65000|
|   106|Frank| 72000|
|   107|Grace| 68000|
|   108|Henry| 52000|
+------+-----+------+



In [12]:
emp_filtered.printSchema()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- department: string (nullable = true)
 |-- gender: string (nullable = true)



In [13]:
from pyspark.sql.functions import when,concat,concat_ws,lit
emp_new = emp_df.withColumn("bonus",col('salary')*0.10)\
                .withColumn("name-dept",concat(col("name"),lit(" "),col("department")))
#or
'''emp_new = emp_df.withColumn("bonus",expr("salary * 0.10 as bonus"))\
                .withColumn("name-dept",expr("name || '-' || department"))\
                .withColumn("gender_short",
                            when(col('gender') == 'Male','M')
                            .when(col('gender') == "Female","F")
                            .otherwise("None"))'''

'emp_new = emp_df.withColumn("bonus",expr("salary * 0.10 as bonus"))                .withColumn("name-dept",expr("name || \'-\' || department"))                .withColumn("gender_short",\n                            when(col(\'gender\') == \'Male\',\'M\')\n                            .when(col(\'gender\') == "Female","F")\n                            .otherwise("None"))'

In [14]:
emp_new.show()

+------+-----+----------+------+---------+---+----------+------+------+-------------+
|emp_id| name|department|salary|     city|age| hire_date|gender| bonus|    name-dept|
+------+-----+----------+------+---------+---+----------+------+------+-------------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|5000.0|     Alice HR|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|6000.0|       Bob IT|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|5500.0|     Cathy HR|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|7000.0|     David IT|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|6500.0|  Eva Finance|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  NULL|7200.0|     Frank IT|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|6800.0|Grace Finance|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|5200.0|     Henry HR|
+------+-----+----------+------+---------+---+--------

In [15]:
emp_new = emp_new.withColumnRenamed("bonus","commission")

In [16]:
emp_new.columns

['emp_id',
 'name',
 'department',
 'salary',
 'city',
 'age',
 'hire_date',
 'gender',
 'commission',
 'name-dept']

In [17]:
emp_new = emp_new.dropna()
emp_new.show()

+------+-----+----------+------+---------+---+----------+------+----------+-------------+
|emp_id| name|department|salary|     city|age| hire_date|gender|commission|    name-dept|
+------+-----+----------+------+---------+---+----------+------+----------+-------------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|    5000.0|     Alice HR|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|    6000.0|       Bob IT|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|    5500.0|     Cathy HR|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|    7000.0|     David IT|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|    6500.0|  Eva Finance|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|    6800.0|Grace Finance|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|    5200.0|     Henry HR|
+------+-----+----------+------+---------+---+----------+------+----------+-------------+



In [18]:
emp_new = emp_new.drop("name-dept","commission")
emp_new.show()


+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
+------+-----+----------+------+---------+---+----------+------+



In [19]:
#converting date
from pyspark.sql.functions import to_date

emp_df = emp_df.withColumn("hire_date",to_date(col('hire_date'),'yyyy-MM-dd'))

In [20]:
emp_df.printSchema()

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- gender: string (nullable = true)



In [21]:
emp_df.show()

+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  NULL|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
+------+-----+----------+------+---------+---+----------+------+



Handling Nulls

In [22]:
emp_df.filter(col('gender').isNull()).show()

+------+-----+----------+------+----+---+----------+------+
|emp_id| name|department|salary|city|age| hire_date|gender|
+------+-----+----------+------+----+---+----------+------+
|   106|Frank|        IT| 72000|Pune| 33|2018-09-12|  NULL|
+------+-----+----------+------+----+---+----------+------+



In [23]:
from pyspark.sql.functions import sum
emp_df.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in emp_df.columns]).show()

+------+----+----------+------+----+---+---------+------+
|emp_id|name|department|salary|city|age|hire_date|gender|
+------+----+----------+------+----+---+---------+------+
|     0|   0|         0|     0|   0|  0|        0|     1|
+------+----+----------+------+----+---+---------+------+



In [24]:
emp_df = emp_df.na.fill("None",'gender')

In [25]:
emp_df.show()

+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
+------+-----+----------+------+---------+---+----------+------+



In [26]:
emp_df.filter(col('gender').isNull()).show()

+------+----+----------+------+----+---+---------+------+
|emp_id|name|department|salary|city|age|hire_date|gender|
+------+----+----------+------+----+---+---------+------+
+------+----+----------+------+----+---+---------+------+



In [27]:
emp_df.orderBy(col('salary').desc()).show()              #by default its ascending

+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
+------+-----+----------+------+---------+---+----------+------+



In [28]:
emp_df.orderBy(col('emp_id'),col('salary').desc()).show()

+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
+------+-----+----------+------+---------+---+----------+------+



In [29]:
emp_df.sort(col('salary').desc()).show()

+------+-----+----------+------+---------+---+----------+------+
|emp_id| name|department|salary|     city|age| hire_date|gender|
+------+-----+----------+------+---------+---+----------+------+
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|
+------+-----+----------+------+---------+---+----------+------+



Simple Aggregation & GroupBy aggregation

In [30]:
from pyspark.sql.functions import avg,count,sum
emp_df.select(
    sum('salary').alias('total_salary'),
    avg('salary').alias('avg_salary')
).show()

+------------+----------+
|total_salary|avg_salary|
+------------+----------+
|      492000|   61500.0|
+------------+----------+



In [31]:

emp_df.groupBy('department').agg(
    count('emp_id').alias('employee count'),
    sum('salary').alias('total_Salary'),
).show()

+----------+--------------+------------+
|department|employee count|total_Salary|
+----------+--------------+------------+
|        HR|             3|      157000|
|        IT|             3|      202000|
|   Finance|             2|      133000|
+----------+--------------+------------+



In [32]:
emp_df.groupBy('department').sum().show()             #sums every column

+----------+-----------+-----------+--------+
|department|sum(emp_id)|sum(salary)|sum(age)|
+----------+-----------+-----------+--------+
|        HR|        312|     157000|      85|
|        IT|        312|     202000|     100|
|   Finance|        212|     133000|      60|
+----------+-----------+-----------+--------+



In [33]:
from pyspark.sql.functions import countDistinct,collect_list,collect_set,first,last,max
emp_df.select(countDistinct('department'),collect_list('department'),collect_set('department')).show(truncate=False)

+--------------------------+------------------------------------------+-----------------------+
|count(DISTINCT department)|collect_list(department)                  |collect_set(department)|
+--------------------------+------------------------------------------+-----------------------+
|3                         |[HR, HR, HR, IT, IT, IT, Finance, Finance]|[HR, Finance, IT]      |
+--------------------------+------------------------------------------+-----------------------+



In [34]:
emp_df.groupBy('department').agg(
    max('salary').alias('highest salary')
).show()

+----------+--------------+
|department|highest salary|
+----------+--------------+
|        HR|         55000|
|        IT|         72000|
|   Finance|         68000|
+----------+--------------+



In [35]:
from pyspark.sql.functions import min

emp_df.select(first("salary").alias('first_value'),last("salary").alias('last_value')).show()            #without order by
emp_df.orderBy('salary').select(first("salary").alias('first_value'),last("salary").alias('last_value')).show()     #with order by

+-----------+----------+
|first_value|last_value|
+-----------+----------+
|      50000|     52000|
+-----------+----------+

+-----------+----------+
|first_value|last_value|
+-----------+----------+
|      50000|     72000|
+-----------+----------+



Joins

In [36]:
df1 = spark.createDataFrame(
    [(1, "Alice"), (2, "Bob"), (3, "Charlie")],
    ["id", "name"]
)

df2 = spark.createDataFrame(
    [(1, "HR"), (2, "IT"), (4, "Finance")],
    ["id", "dept"]
)

df3 = spark.createDataFrame(
    [(1, "HR"), (2, "IT"), (4, "Finance")],
    ["empid", "dept"]
)

In [37]:
#inner-join
df1.join(df2,on = 'id').show()

+---+-----+----+
| id| name|dept|
+---+-----+----+
|  1|Alice|  HR|
|  2|  Bob|  IT|
+---+-----+----+



In [38]:
#different column names
from pyspark.sql.functions import broadcast
df1.join(broadcast(df3),df1.id == df3.empid).explain()
df1.join(df3,df1.id == df3.empid).drop('empid').show()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [id#652L], [empid#656L], Inner, BuildRight, false
   :- Filter isnotnull(id#652L)
   :  +- Scan ExistingRDD[id#652L,name#653]
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=740]
      +- Filter isnotnull(empid#656L)
         +- Scan ExistingRDD[empid#656L,dept#657]


+---+-----+----+
| id| name|dept|
+---+-----+----+
|  1|Alice|  HR|
|  2|  Bob|  IT|
+---+-----+----+



In [39]:
#left join
df1.join(df2,on = 'id',how='left_outer').show()

+---+-------+----+
| id|   name|dept|
+---+-------+----+
|  1|  Alice|  HR|
|  3|Charlie|NULL|
|  2|    Bob|  IT|
+---+-------+----+



In [40]:
#right join
df1.join(df2,'id','right').show()

+---+-----+-------+
| id| name|   dept|
+---+-----+-------+
|  1|Alice|     HR|
|  2|  Bob|     IT|
|  4| NULL|Finance|
+---+-----+-------+



In [41]:
#left_semi
df1.join(df2,'id','left_semi').show()              #columns of left table and only rows which has matching records in right table

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [42]:
#left_anti
df1.join(df2,'id','left_anti').show()          #columns of left table and only rows which has no matching records in right table

+---+-------+
| id|   name|
+---+-------+
|  3|Charlie|
+---+-------+



In [43]:
from pyspark.sql.functions import rand

df1 = df1.withColumn("salt", (rand()*10).cast("int"))
df1.show()

+---+-------+----+
| id|   name|salt|
+---+-------+----+
|  1|  Alice|   5|
|  2|    Bob|   0|
|  3|Charlie|   9|
+---+-------+----+



In [44]:
from pyspark.sql.functions import explode, array

df2 = df2.withColumn("salt", explode(array([lit(i) for i in range(10)])))
df2.head(10)

[Row(id=1, dept='HR', salt=0),
 Row(id=1, dept='HR', salt=1),
 Row(id=1, dept='HR', salt=2),
 Row(id=1, dept='HR', salt=3),
 Row(id=1, dept='HR', salt=4),
 Row(id=1, dept='HR', salt=5),
 Row(id=1, dept='HR', salt=6),
 Row(id=1, dept='HR', salt=7),
 Row(id=1, dept='HR', salt=8),
 Row(id=1, dept='HR', salt=9)]

In [45]:
df1.join(df2,['id','salt']).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id#652L, salt#712, name#653, dept#655]
   +- SortMergeJoin [id#652L, salt#712], [id#654L, salt#724], Inner
      :- Sort [id#652L ASC NULLS FIRST, salt#712 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id#652L, salt#712, 200), ENSURE_REQUIREMENTS, [plan_id=1393]
      :     +- Filter (isnotnull(id#652L) AND isnotnull(salt#712))
      :        +- Project [id#652L, name#653, cast((rand(-3311180250854119689) * 10.0) as int) AS salt#712]
      :           +- Scan ExistingRDD[id#652L,name#653]
      +- Sort [id#654L ASC NULLS FIRST, salt#724 ASC NULLS FIRST], false, 0
         +- Exchange hashpartitioning(id#654L, salt#724, 200), ENSURE_REQUIREMENTS, [plan_id=1394]
            +- Generate explode([0,1,2,3,4,5,6,7,8,9]), [id#654L, dept#655], false, [salt#724]
               +- Filter isnotnull(id#654L)
                  +- Scan ExistingRDD[id#654L,dept#655]




Window functions

In [46]:
emp_df.columns

['emp_id',
 'name',
 'department',
 'salary',
 'city',
 'age',
 'hire_date',
 'gender']

In [47]:
#highest salary in each department
from pyspark.sql.window import Window
from pyspark.sql.functions import max

window_spec = Window.partitionBy('department').orderBy(col('salary').desc())
max_salary_df = emp_df.withColumn('highest_Salary',max('salary').over(window_spec))
max_salary_df.show()


+------+-----+----------+------+---------+---+----------+------+--------------+
|emp_id| name|department|salary|     city|age| hire_date|gender|highest_Salary|
+------+-----+----------+------+---------+---+----------+------+--------------+
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|         68000|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|         68000|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|         55000|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|         55000|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|         55000|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|         72000|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|         72000|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|         72000|
+------+-----+----------+------+---------+---+----------+------+--------------+



In [48]:
highest_sal_df = emp_df.withColumn('highest_salary',max('salary').over(Window.partitionBy('department')))

In [49]:
highest_sal_df.show()

+------+-----+----------+------+---------+---+----------+------+--------------+
|emp_id| name|department|salary|     city|age| hire_date|gender|highest_salary|
+------+-----+----------+------+---------+---+----------+------+--------------+
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|         68000|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|         68000|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|         55000|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|         55000|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|         55000|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|         72000|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|         72000|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|         72000|
+------+-----+----------+------+---------+---+----------+------+--------------+



In [50]:
from pyspark.sql.functions import row_number
second_highest_sal_df = emp_df.withColumn('rnk',row_number().over(window_spec))
second_highest_sal_df.where(col('rnk') == 2).show()

+------+-----+----------+------+---------+---+----------+------+---+
|emp_id| name|department|salary|     city|age| hire_date|gender|rnk|
+------+-----+----------+------+---------+---+----------+------+---+
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|  2|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|  2|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|  2|
+------+-----+----------+------+---------+---+----------+------+---+



In [51]:
#using expr
second_highest_sal_df1 = emp_df.withColumn('rnk',expr("row_number() over(partition by department order by salary desc)")).where(col('rnk') == 2)
second_highest_sal_df1.show()

+------+-----+----------+------+---------+---+----------+------+---+
|emp_id| name|department|salary|     city|age| hire_date|gender|rnk|
+------+-----+----------+------+---------+---+----------+------+---+
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|  2|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|  2|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|  2|
+------+-----+----------+------+---------+---+----------+------+---+



In [52]:
#top 3
top_3 = emp_df.withColumn('rnk',row_number().over(window_spec))
top_3.filter("rnk <= 3").show()

+------+-----+----------+------+---------+---+----------+------+---+
|emp_id| name|department|salary|     city|age| hire_date|gender|rnk|
+------+-----+----------+------+---------+---+----------+------+---+
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|  1|
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|  2|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|  1|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|  2|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|  3|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|  1|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|  2|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|  3|
+------+-----+----------+------+---------+---+----------+------+---+



In [53]:
#cume_dist and percent_rank
from pyspark.sql.functions import cume_dist,percent_rank,round,concat
emp_cumulative = emp_df.withColumn('cumulative',concat(round(cume_dist().over(Window.partitionBy('department').orderBy('salary')),2),lit('%')))
emp_cumulative.show()

+------+-----+----------+------+---------+---+----------+------+----------+
|emp_id| name|department|salary|     city|age| hire_date|gender|cumulative|
+------+-----+----------+------+---------+---+----------+------+----------+
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|      0.5%|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|      1.0%|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|     0.33%|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|     0.67%|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|      1.0%|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|     0.33%|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|     0.67%|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|      1.0%|
+------+-----+----------+------+---------+---+----------+------+----------+



In [54]:
emp_percent = emp_df.withColumn('cumulative',concat(round(percent_rank().over(Window.partitionBy('department').orderBy('salary')),2),lit('%')))
emp_percent.show()

+------+-----+----------+------+---------+---+----------+------+----------+
|emp_id| name|department|salary|     city|age| hire_date|gender|cumulative|
+------+-----+----------+------+---------+---+----------+------+----------+
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|      0.0%|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|      1.0%|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|      0.0%|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|      0.5%|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|      1.0%|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|      0.0%|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|      0.5%|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|      1.0%|
+------+-----+----------+------+---------+---+----------+------+----------+



In [55]:
cum_window = Window.partitionBy('department').orderBy('salary').rowsBetween(Window.unboundedPreceding,Window.currentRow)
cum_df = emp_df.withColumn('cumulative',round(sum('salary').over(cum_window)/sum('salary').over(Window.partitionBy('department')),2))
cum_df.show()

+------+-----+----------+------+---------+---+----------+------+----------+
|emp_id| name|department|salary|     city|age| hire_date|gender|cumulative|
+------+-----+----------+------+---------+---+----------+------+----------+
|   105|  Eva|   Finance| 65000|   Mumbai| 29|2021-06-18|  Male|      0.49|
|   107|Grace|   Finance| 68000|Bangalore| 31|2020-12-01|Female|       1.0|
|   101|Alice|        HR| 50000|     Pune| 28|2021-03-15|  Male|      0.32|
|   108|Henry|        HR| 52000|   Mumbai| 27|2023-04-22|  Male|      0.65|
|   103|Cathy|        HR| 55000|     Pune| 30|2022-01-05|Female|       1.0|
|   102|  Bob|        IT| 60000|   Mumbai| 32|2020-07-10|  Male|       0.3|
|   104|David|        IT| 70000|Bangalore| 35|2019-11-20|Female|      0.64|
|   106|Frank|        IT| 72000|     Pune| 33|2018-09-12|  None|       1.0|
+------+-----+----------+------+---------+---+----------+------+----------+



In [56]:
emp_df.select([
    count(when(col(c).isNull(),c)).alias(c) for c in emp_df.columns]).show()

+------+----+----------+------+----+---+---------+------+
|emp_id|name|department|salary|city|age|hire_date|gender|
+------+----+----------+------+----+---+---------+------+
|     0|   0|         0|     0|   0|  0|        0|     0|
+------+----+----------+------+----+---+---------+------+



In [57]:
from pyspark.sql.functions import isnan,isnotnull

cum_df.filter(isnan('cumulative')).count()
cum_df.filter(col('cumulative').isNaN()).count()

0

In [58]:
#calculating median
from pyspark.sql.functions import percentile_approx
median_salary = emp_df.select(
    percentile_approx("salary", 0.5).alias("median")
).first()["median"]
median_salary

60000

In [59]:
#calculating mode
emp_df.groupby("department").count().orderBy("count",ascending=False).limit(1).show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    3|
+----------+-----+



In [60]:
emp_df.dtypes

[('emp_id', 'bigint'),
 ('name', 'string'),
 ('department', 'string'),
 ('salary', 'bigint'),
 ('city', 'string'),
 ('age', 'bigint'),
 ('hire_date', 'date'),
 ('gender', 'string')]

In [61]:
emp_df.rdd.getNumPartitions()

2

In [62]:
emp_df = emp_df.repartition(10)

In [63]:
emp_df.rdd.getNumPartitions()

10

UDF


In [64]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import IntegerType, StringType

In [65]:
df = spark.createDataFrame([(1,), (2,), (3,)], ["num"])
df.show()

+---+
|num|
+---+
|  1|
|  2|
|  3|
+---+



In [66]:
#way1
@udf(returnType = IntegerType())
def square(x):
    if x is not None:
      return x*x

df = df.withColumn("square",square(col("num")))
df.show()

+---+------+
|num|square|
+---+------+
|  1|     1|
|  2|     4|
|  3|     9|
+---+------+



In [67]:
#way2
def square(x):
    if x is not None:
      return x*x
square_udf = udf(square,IntegerType())
df = df.withColumn("square",square_udf(col("num")))
df.show()

+---+------+
|num|square|
+---+------+
|  1|     1|
|  2|     4|
|  3|     9|
+---+------+



In [68]:
#using pandas_udf
import pandas as pd
@pandas_udf(IntegerType())
def double(x:pd.Series) -> pd.Series:
    if x is not None:
      return x*2
df = df.withColumn("double",double(col("num")))
df.show()

+---+------+------+
|num|square|double|
+---+------+------+
|  1|     1|     2|
|  2|     4|     4|
|  3|     9|     6|
+---+------+------+



In [69]:
spark.udf.register('double_udf',double)

df.createOrReplaceTempView('numbers')
spark.sql("select num, double_udf(num) as Double from numbers").show()

+---+------+
|num|Double|
+---+------+
|  1|     2|
|  2|     4|
|  3|     6|
+---+------+



GroupedMap udf

In [70]:
from pyspark.sql.types import StructType, StructField
schema = StructType([
    StructField("id", IntegerType()),
    StructField("group", StringType()),
    StructField("value", IntegerType()),
    StructField("value_sum", IntegerType())
])


In [71]:
data = [(1, "A", 10),
        (2, "A", 20),
        (3, "B", 30),
        (4, "B", 40)]

df1 = spark.createDataFrame(data, ["id", "group", "value"])
df1.show()

+---+-----+-----+
| id|group|value|
+---+-----+-----+
|  1|    A|   10|
|  2|    A|   20|
|  3|    B|   30|
|  4|    B|   40|
+---+-----+-----+



In [72]:
from pyspark.sql.functions import PandasUDFType
@pandas_udf(schema,functionType=PandasUDFType.GROUPED_MAP)
def sum_values(df:pd.DataFrame) -> pd.DataFrame:
  df['value_sum'] = df['value'].sum()
  return df

In [73]:
df1.groupBy('group').apply(sum_values).show()

/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/group_ops.py:122: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


+---+-----+-----+---------+
| id|group|value|value_sum|
+---+-----+-----+---------+
|  1|    A|   10|       30|
|  2|    A|   20|       30|
|  3|    B|   30|       70|
|  4|    B|   40|       70|
+---+-----+-----+---------+



Date Functions

In [74]:
from pyspark.sql.functions import *

In [75]:
date_df = emp_df.select('hire_date')
date_df.show()

+----------+
| hire_date|
+----------+
|2020-07-10|
|2021-03-15|
|2022-01-05|
|2023-04-22|
|2021-06-18|
|2018-09-12|
|2020-12-01|
|2019-11-20|
+----------+



year(),month(),dayofmonth()

In [76]:
date_df.select(dayofweek("hire_date")).show()     #7-saturday,2-Monday,1-Sunday

+--------------------+
|dayofweek(hire_date)|
+--------------------+
|                   6|
|                   2|
|                   4|
|                   7|
|                   6|
|                   4|
|                   3|
|                   4|
+--------------------+



In [77]:
date_df.select("hire_date",date_add("hire_date",5).alias('after5days')).show()

+----------+----------+
| hire_date|after5days|
+----------+----------+
|2020-07-10|2020-07-15|
|2021-03-15|2021-03-20|
|2022-01-05|2022-01-10|
|2023-04-22|2023-04-27|
|2021-06-18|2021-06-23|
|2018-09-12|2018-09-17|
|2020-12-01|2020-12-06|
|2019-11-20|2019-11-25|
+----------+----------+



In [78]:
date_df.select("hire_date",date_sub("hire_date",30).alias('before30days')).show()

+----------+------------+
| hire_date|before30days|
+----------+------------+
|2020-07-10|  2020-06-10|
|2021-03-15|  2021-02-13|
|2022-01-05|  2021-12-06|
|2023-04-22|  2023-03-23|
|2021-06-18|  2021-05-19|
|2018-09-12|  2018-08-13|
|2020-12-01|  2020-11-01|
|2019-11-20|  2019-10-21|
+----------+------------+



In [79]:
date_df.select("hire_date",trunc('hire_date','week')).show()      #similarily date_trunc(format, "timestamp") is used for timestamp

+----------+----------------------+
| hire_date|trunc(hire_date, week)|
+----------+----------------------+
|2020-07-10|            2020-07-06|
|2021-03-15|            2021-03-15|
|2022-01-05|            2022-01-03|
|2023-04-22|            2023-04-17|
|2021-06-18|            2021-06-14|
|2018-09-12|            2018-09-10|
|2020-12-01|            2020-11-30|
|2019-11-20|            2019-11-18|
+----------+----------------------+



String functions

In [102]:
name_df = emp_df.select('name')
name_df.show()

+-----+
| name|
+-----+
|Alice|
|David|
|Cathy|
|  Eva|
|Grace|
|Henry|
|Frank|
|  Bob|
+-----+



In [103]:
name_df.select(upper('name')).show()

+-----------+
|upper(name)|
+-----------+
|      ALICE|
|      DAVID|
|      CATHY|
|        EVA|
|      GRACE|
|      HENRY|
|      FRANK|
|        BOB|
+-----------+



In [104]:
name_df.withColumn('name_length',length(col('name'))).show()

+-----+-----------+
| name|name_length|
+-----+-----------+
|Alice|          5|
|David|          5|
|Cathy|          5|
|  Eva|          3|
|Grace|          5|
|Henry|          5|
|Frank|          5|
|  Bob|          3|
+-----+-----------+



In [106]:
name_df = name_df.withColumn('length',lit(2))
name_df.show()

+-----+------+
| name|length|
+-----+------+
|Alice|     2|
|David|     2|
|Cathy|     2|
|  Eva|     2|
|Grace|     2|
|Henry|     2|
|Frank|     2|
|  Bob|     2|
+-----+------+



In [117]:
name_df.select(substr('name',name_df.length).alias("name_substr")).show()

+-----------+
|name_substr|
+-----------+
|       lice|
|       avid|
|       athy|
|         va|
|       race|
|       enry|
|       rank|
|         ob|
+-----------+



In [125]:
#name_df.select(expr("replace(name,'a','x')").alias("after_replace")).show()
#or
name_df.select(translate("name", "a", "x").alias("after_replace")).show()

+-------------+
|after_replace|
+-------------+
|        Alice|
|        Dxvid|
|        Cxthy|
|          Evx|
|        Grxce|
|        Henry|
|        Frxnk|
|          Bob|
+-------------+



In [128]:
df = spark.createDataFrame([('100-200',)], ['str'])
df.select('*',regexp_extract('str', r'(\d+)-(\d+)', 1)).show()
df.select('*',regexp_extract('str', r'(\d+)-(\d+)', 2)).show()

+-------+-----------------------------------+
|    str|regexp_extract(str, (\d+)-(\d+), 1)|
+-------+-----------------------------------+
|100-200|                                100|
+-------+-----------------------------------+

+-------+-----------------------------------+
|    str|regexp_extract(str, (\d+)-(\d+), 2)|
+-------+-----------------------------------+
|100-200|                                200|
+-------+-----------------------------------+



In [131]:
name_df.filter(col('name').like('A%')).show()

+-----+------+
| name|length|
+-----+------+
|Alice|     2|
+-----+------+



In [137]:
#name_df.filter(col('name').rlike('^B')).show()
#or
name_df.filter(rlike('name',lit('^B'))).show()

+----+------+
|name|length|
+----+------+
| Bob|     2|
+----+------+



In [142]:
name_df.withColumn('name',rpad('name',6,'#')).show()

+------+------+
|  name|length|
+------+------+
|Alice#|     2|
|David#|     2|
|Cathy#|     2|
|Eva###|     2|
|Grace#|     2|
|Henry#|     2|
|Frank#|     2|
|Bob###|     2|
+------+------+



In [143]:
name_df.withColumn('length',lpad('length',5,0)).show()

+-----+------+
| name|length|
+-----+------+
|Alice| 00002|
|David| 00002|
|Cathy| 00002|
|  Eva| 00002|
|Grace| 00002|
|Henry| 00002|
|Frank| 00002|
|  Bob| 00002|
+-----+------+



Complex Types - ArrayType, MapType, StructType

In [153]:
data = [
    (1, ["apple", "banana"]),
    (2, ["orange",'kiwi']),
    (3, [])
]

fruits = spark.createDataFrame(data, ["id", "name"])
fruits.show()

+---+---------------+
| id|           name|
+---+---------------+
|  1|[apple, banana]|
|  2| [orange, kiwi]|
|  3|             []|
+---+---------------+



In [154]:
fruits.select(explode("name")).show()

+------+
|   col|
+------+
| apple|
|banana|
|orange|
|  kiwi|
+------+



In [155]:
fruits.select(explode_outer("name")).show()

+------+
|   col|
+------+
| apple|
|banana|
|orange|
|  kiwi|
|  NULL|
+------+



In [156]:
fruits.select(posexplode("name")).show()

+---+------+
|pos|   col|
+---+------+
|  0| apple|
|  1|banana|
|  0|orange|
|  1|  kiwi|
+---+------+



In [157]:
fruits.select(explode(sort_array("name"))).show()

+------+
|   col|
+------+
| apple|
|banana|
|  kiwi|
|orange|
+------+



In [161]:
fruits.select(array_remove("name",'kiwi')).show()

+------------------------+
|array_remove(name, kiwi)|
+------------------------+
|         [apple, banana]|
|                [orange]|
|                      []|
+------------------------+



In [165]:
fruits.filter(array_contains('name',"banana")).show()
fruits.filter(array_contains('name',"pineapple")).show()

+---+---------------+
| id|           name|
+---+---------------+
|  1|[apple, banana]|
+---+---------------+

+---+----+
| id|name|
+---+----+
+---+----+



In [166]:
fruits.select(size('name')).show()

+----------+
|size(name)|
+----------+
|         2|
|         2|
|         0|
+----------+



StructType

In [224]:
schema = StructType([
    StructField("id", IntegerType()),
    StructField("info", StructType([
        StructField("name", StringType()),
        StructField("age", IntegerType())
    ]))
])
data_df = spark.createDataFrame([
    (1, ("John", 25)), (2, ("Alice", 30))],schema)
data_df.show()

+---+-----------+
| id|       info|
+---+-----------+
|  1| {John, 25}|
|  2|{Alice, 30}|
+---+-----------+



In [225]:
data_df.select("id","info.name").show()

+---+-----+
| id| name|
+---+-----+
|  1| John|
|  2|Alice|
+---+-----+



In [226]:
data_df = data_df.withColumn('name',col('info').withField('name',lit('Mike')).alias('name'))

In [227]:
data_df.show()

+---+-----------+----------+
| id|       info|      name|
+---+-----------+----------+
|  1| {John, 25}|{Mike, 25}|
|  2|{Alice, 30}|{Mike, 30}|
+---+-----------+----------+



In [228]:
data_df = data_df.withColumn('info',col('info').dropFields("name"))
data_df.show()

+---+----+----------+
| id|info|      name|
+---+----+----------+
|  1|{25}|{Mike, 25}|
|  2|{30}|{Mike, 30}|
+---+----+----------+



In [230]:
data_df.select(to_json('name').alias('json')).show(truncate= False)

+------------------------+
|json                    |
+------------------------+
|{"name":"Mike","age":25}|
|{"name":"Mike","age":30}|
+------------------------+



MapType

In [235]:
marks_df = spark.createDataFrame([
    (1, {"Maths": 85, "Science": 90}),
    (2, {"Maths": 75, "Science": 80}),
    (3, {"Maths": 90, "Science": 85})
], ["id", "marks"])
marks_df.show(truncate=False)

+---+----------------------------+
|id |marks                       |
+---+----------------------------+
|1  |{Science -> 90, Maths -> 85}|
|2  |{Science -> 80, Maths -> 75}|
|3  |{Science -> 85, Maths -> 90}|
+---+----------------------------+



In [236]:
marks_df.select("marks.Science").show()

+-------+
|Science|
+-------+
|     90|
|     80|
|     85|
+-------+



In [237]:
df_m = spark.createDataFrame([(1, 90, 80)], ["id", "math", "english"])

df_m.withColumn(
    "marks",
    create_map(lit("math"), col("math"), lit("english"), col("english"))
).show(truncate=False)

+---+----+-------+---------------------------+
|id |math|english|marks                      |
+---+----+-------+---------------------------+
|1  |90  |80     |{math -> 90, english -> 80}|
+---+----+-------+---------------------------+



In [239]:
marks_df.select(map_keys('marks')).show()

+----------------+
| map_keys(marks)|
+----------------+
|[Science, Maths]|
|[Science, Maths]|
|[Science, Maths]|
+----------------+



In [240]:
marks_df.select(map_values('marks')).show()

+-----------------+
|map_values(marks)|
+-----------------+
|         [90, 85]|
|         [80, 75]|
|         [85, 90]|
+-----------------+



In [243]:
marks_df.select(element_at("marks",'English')).show()
marks_df.select(element_at("marks",'Maths')).show()

+--------------------------+
|element_at(marks, English)|
+--------------------------+
|                      NULL|
|                      NULL|
|                      NULL|
+--------------------------+

+------------------------+
|element_at(marks, Maths)|
+------------------------+
|                      85|
|                      75|
|                      90|
+------------------------+



In [244]:
marks_df.select(size('marks')).show()

+-----------+
|size(marks)|
+-----------+
|          2|
|          2|
|          2|
+-----------+



In [246]:
marks_df.select(explode('marks')).show()

+-------+-----+
|    key|value|
+-------+-----+
|Science|   90|
|  Maths|   85|
|Science|   80|
|  Maths|   75|
|Science|   85|
|  Maths|   90|
+-------+-----+



In [252]:
marks_df.select(map_filter("marks",lambda k,v : v > 85).alias('marks > 80')).show(truncate=False)

+---------------+
|marks > 80     |
+---------------+
|{Science -> 90}|
|{}             |
|{Maths -> 90}  |
+---------------+

